In [87]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape
from shapely import wkt
import warnings
# filter all warnings
warnings.filterwarnings("ignore")

In [88]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)



In [89]:
with open('data/Bus/AMB.json') as f:
    data = json.load(f)


stops_amb = []

for collection in data:
    for feature in collection["features"]:
        props = feature["properties"]
        geom = feature["geometry"]
        if geom["type"] == "Point":
            stops_amb.append({
                "code": props["COD_EMT"],
                "name": props["NOMBRE"],
                "geometry": shape(geom)
            })

geo_df_stops_amb = gpd.GeoDataFrame(stops_amb,geometry="geometry", crs="EPSG:4326")


buffered = neigh.copy()
buffered.to_crs('EPSG:25831', inplace=True)
buffered["geometry"] = buffered.geometry.buffer(2500)
geo_df_stops_amb.to_crs('EPSG:25831', inplace=True)
geo_df_stops_amb = gpd.sjoin(
    geo_df_stops_amb,
    buffered[["geometry"]],
    predicate="within",
    how="inner",
).drop(columns="index_right")
geo_df_stops_amb.to_crs('EPSG:4326', inplace=True)
geo_df_stops_amb.drop_duplicates(subset=["code"], inplace=True)
geo_df_stops_amb

,code,name,geometry
223,200103,Riera Bonet,POINT (2.02769 41.40477)
267,200152,Esglesia St. Bartomeu,POINT (2.03807 41.42635)
268,200178,Esglesia St. Bartomeu,POINT (2.03821 41.42648)
277,200153,Ctra. de Vallvidrera,POINT (2.04046 41.42781)
300,105724,Roserar Dot i Camprubi,POINT (2.04579 41.39055)
...,...,...,...
12809,107905,Poliesportiu Marina Besos,POINT (2.23174 41.42347)
12813,102334,Salvador Espriu,POINT (2.23246 41.44564)
12815,110367,Tortosa - Guifre,POINT (2.23514 41.43701)
12823,107908,La Mora,POINT (2.23756 41.43223)


In [90]:
# delete all the zeros before the bus stop code
geo_df_stops_amb["code"] = geo_df_stops_amb["code"].apply(lambda x: x.lstrip("0")).astype(int)
geo_df_stops_amb

,code,name,geometry
223,200103,Riera Bonet,POINT (2.02769 41.40477)
267,200152,Esglesia St. Bartomeu,POINT (2.03807 41.42635)
268,200178,Esglesia St. Bartomeu,POINT (2.03821 41.42648)
277,200153,Ctra. de Vallvidrera,POINT (2.04046 41.42781)
300,105724,Roserar Dot i Camprubi,POINT (2.04579 41.39055)
...,...,...,...
12809,107905,Poliesportiu Marina Besos,POINT (2.23174 41.42347)
12813,102334,Salvador Espriu,POINT (2.23246 41.44564)
12815,110367,Tortosa - Guifre,POINT (2.23514 41.43701)
12823,107908,La Mora,POINT (2.23756 41.43223)


In [91]:
bus_stops_w_line = pd.read_excel("Data/Bus/bus_stops_w_lines.xlsx")
bus_stops_w_line.rename(columns={"Código de parada / Codi de parada": "code",'Municipio / Municipi': 'municipi','Líneas / Línies': 'lines'}, inplace=True)
bus_stops_w_line = bus_stops_w_line[["code", "municipi", "lines"]]
geo_df_stops_amb = geo_df_stops_amb.merge(bus_stops_w_line, on="code", how="left")
print(f"Number of bus stops without lines: {geo_df_stops_amb['lines'].isna().sum()}")
missing_stops = geo_df_stops_amb[geo_df_stops_amb["lines"].isna()]
geo_df_stops_amb = geo_df_stops_amb[geo_df_stops_amb["lines"].notnull()]
geo_df_stops_amb["lines"] = geo_df_stops_amb["lines"].apply(lambda x: [line.strip() for line in x.split("-")])
geo_df_stops_amb = geo_df_stops_amb.explode("lines", ignore_index=True)
# delete lines that start with N
geo_df_stops_amb = geo_df_stops_amb[~geo_df_stops_amb["lines"].str.startswith("N")]
geo_df_stops_amb['lines'].nunique(), geo_df_stops_amb['code'].nunique()


Number of bus stops without lines: 1326


(144, 2111)

In [92]:
missing_stops

,code,name,geometry,municipi,lines
0,200103,Riera Bonet,POINT (2.02769 41.40477),NaN,NaN
1,200152,Esglesia St. Bartomeu,POINT (2.03807 41.42635),NaN,NaN
2,200178,Esglesia St. Bartomeu,POINT (2.03821 41.42648),NaN,NaN
3,200153,Ctra. de Vallvidrera,POINT (2.04046 41.42781),NaN,NaN
4,105724,Roserar Dot i Camprubi,POINT (2.04579 41.39055),NaN,NaN
...,...,...,...,...,...
3457,107905,Poliesportiu Marina Besos,POINT (2.23174 41.42347),NaN,NaN
3458,102334,Salvador Espriu,POINT (2.23246 41.44564),NaN,NaN
3459,110367,Tortosa - Guifre,POINT (2.23514 41.43701),NaN,NaN
3460,107908,La Mora,POINT (2.23756 41.43223),NaN,NaN
